# Séance 6 · Exercices — Modéliser : ton premier modèle de machine learning · ⭐⭐

**Niveau : ⭐⭐ Intermédiaire**

**Niveau de la séance : ⭐⭐ Intermédiaire** · chaque exercice porte son propre niveau (⭐ Débutant · ⭐⭐ Intermédiaire · ⭐⭐⭐ Avancé).

- Comment travailler : lis l'énoncé, code dans la cellule « À toi », lance la cellule de vérification (✅ / ❌), et n'ouvre la solution qu'après avoir vraiment essayé.
- Ce notebook tourne dans **Google Colab** : rien à installer.
- Clique sur une cellule et fais `Maj + Entrée` pour l'exécuter. Fais les exercices dans l'ordre : certains réutilisent les variables des précédents.


## Préparation

Mêmes données que la leçon : les 344 **manchots** de Palmer (déjà nettoyés : les lignes incomplètes sont retirées, il en reste 333) et les 800 **Pokémon**. La boîte à outils : **scikit-learn** (`sklearn`), déjà installée dans Colab.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, mean_absolute_error

URL_MANCHOTS = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
URL_POKEMON = "https://gist.githubusercontent.com/armgilles/194bcff35001e7eb53a2a8b441e8b2c6/raw/92200bc0a673d5ce2110aaad4544ed6c4010f687/pokemon.csv"
try:
    manchots = pd.read_csv(URL_MANCHOTS).dropna().reset_index(drop=True)   # un modèle n'aime pas les cases vides
    pokemon = pd.read_csv(URL_POKEMON)
    print("Manchots :", manchots.shape, "· Pokémon :", pokemon.shape)
except Exception as erreur:
    print("Pas de réseau ? Impossible de charger les fichiers :", erreur)

stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]
manchots.head(3)

def verifier(nom, condition):
    """Affiche ✅ ou ❌ sans jamais lever d'exception (condition = booléen, ou fonction sans argument)."""
    try:
        ok = bool(condition() if callable(condition) else condition)
    except Exception as erreur:
        print(f"❌ {nom} : erreur pendant la vérification → {erreur}")
        return False
    print(f"✅ {nom}" if ok else f"❌ {nom} : pas encore, réessaie !")
    return ok


## Exercice 1 ⭐ · Classification ou régression ?

Pour chaque question, la machine doit-elle deviner une **catégorie** (classification, `"C"`) ou un **nombre**
(régression, `"R"`) ? Remplace les `"?"` du dictionnaire `reponses`.

Résultat attendu : 6 bonnes réponses sur 6.

<details><summary>Indice</summary>

Une catégorie se choisit dans une liste (une espèce, un type, oui / non) ; un nombre se mesure (des degrés, des likes, des euros).

</details>

In [ ]:
# À toi : "C" (classification) ou "R" (régression)
reponses = {
    "temperature_demain": "?",       # deviner la température de demain
    "espece_manchot": "?",           # deviner l'espèce d'un manchot
    "mail_important_ou_pas": "?",    # deviner si un mail est important
    "nombre_de_likes": "?",          # deviner combien de likes aura une photo
    "type_pokemon": "?",             # deviner le type d'un Pokémon
    "prix_voiture_occasion": "?",    # deviner le prix d'une voiture d'occasion
}
print(reponses)

In [ ]:
corrige_1 = {"temperature_demain": "R", "espece_manchot": "C", "mail_important_ou_pas": "C", "nombre_de_likes": "R", "type_pokemon": "C", "prix_voiture_occasion": "R"}
score_1 = sum(reponses.get(k) == v for k, v in corrige_1.items())
print("Score :", score_1, "/ 6")
verifier("Exercice 1 · classification ou régression", score_1 == 6)

<details><summary>Solution</summary>

```python
reponses = {
    "temperature_demain": "R",
    "espece_manchot": "C",
    "mail_important_ou_pas": "C",
    "nombre_de_likes": "R",
    "type_pokemon": "C",
    "prix_voiture_occasion": "R",
}
print(reponses)
```

</details>

## Exercice 2 ⭐ · X et y

Prépare les données des manchots : `colonnes` (les 4 mesures : longueur du bec, épaisseur du bec, longueur de la
nageoire, poids), `X` (ce que la machine voit) et `y` (ce qu'elle doit deviner : `species`). Compte `nb_especes`.

Résultat attendu : `X` a 333 lignes et 4 colonnes, `y` contient 3 espèces.

<details><summary>Indice</summary>

Les colonnes s'appellent `bill_length_mm`, `bill_depth_mm`, `flipper_length_mm`, `body_mass_g` ; `y.nunique()`.

</details>

In [ ]:
# À toi
colonnes = []
X = None
y = None
nb_especes = None
print("X :", None if X is None else X.shape, "· espèces :", nb_especes)

In [ ]:
verifier("Exercice 2 · X", isinstance(X, pd.DataFrame) and X.shape == (333, 4) and set(X.columns) == {"bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"})
verifier("Exercice 2 · y", y is not None and set(y.unique()) == {"Adelie", "Chinstrap", "Gentoo"})
verifier("Exercice 2 · nombre d'espèces", nb_especes == 3)

<details><summary>Solution</summary>

```python
colonnes = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
X = manchots[colonnes]
y = manchots["species"]
nb_especes = y.nunique()
print("X :", X.shape, "· espèces :", nb_especes)
```

</details>

## Exercice 3 ⭐ · Cacher une partie des données

Coupe `X` et `y` en deux avec `train_test_split` : 25 % cachés pour le test, `random_state=42` pour que tout le monde
ait la même coupe. Calcule `part_test` (la part des données cachées, arrondie à 2 décimales).

Résultat attendu : 249 manchots pour apprendre, 84 cachés.

<details><summary>Indice</summary>

`X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)`.

</details>

In [ ]:
# À toi
X_train, X_test, y_train, y_test = None, None, None, None
part_test = None
print("Apprentissage :", None if X_train is None else len(X_train), "· test :", None if X_test is None else len(X_test), "· part test :", part_test)

In [ ]:
verifier("Exercice 3 · la coupe", X_train is not None and len(X_train) == 249 and len(X_test) == 84 and len(y_test) == 84)
verifier("Exercice 3 · même coupe que tout le monde", X_test is not None and list(X_test.index[:3]) == [25, 309, 73])
verifier("Exercice 3 · part test", part_test == 0.25)

<details><summary>Solution</summary>

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
part_test = round(len(X_test) / len(X), 2)
print("Apprentissage :", len(X_train), "· test :", len(X_test), "· part test :", part_test)
```

</details>

## Exercice 4 ⭐ · Le modèle bête

Avant tout modèle, la barre à battre : un « modèle bête » qui répond toujours l'espèce la plus fréquente du train.
Trouve `espece_frequente` et calcule `precision_bete` (la part de bonnes réponses sur les données de **test**).

Résultat attendu : un peu moins d'une bonne réponse sur deux.

<details><summary>Indice</summary>

`y_train.value_counts().index[0]` ; puis `(y_test == espece_frequente).mean()`.

</details>

In [ ]:
# À toi
espece_frequente = None
precision_bete = None
print("Le modèle bête répond toujours", espece_frequente, "→ précision :", precision_bete)

In [ ]:
verifier("Exercice 4 · espèce la plus fréquente", espece_frequente == "Adelie")
verifier("Exercice 4 · précision du modèle bête", precision_bete is not None and round(float(precision_bete), 2) == 0.48)

<details><summary>Solution</summary>

```python
espece_frequente = y_train.value_counts().index[0]
precision_bete = (y_test == espece_frequente).mean()
print("Le modèle bête répond toujours", espece_frequente, "→ précision :", precision_bete)
```

</details>

## Exercice 5 ⭐⭐ · Ton premier arbre

Entraîne `arbre`, un `DecisionTreeClassifier` de profondeur 3 (`random_state=42`), fais-le prédire sur les données
cachées (`predictions`) et mesure `precision_arbre`. Puis trouve `premiere_question` : le nom de la colonne que l'arbre
utilise dans sa toute première question (à lire sur `plot_tree`, ou dans `arbre.tree_.feature[0]`). Affiche l'arbre.

Résultat attendu : une précision bien au-dessus du modèle bête, et une première question sur la nageoire.

<details><summary>Indice</summary>

`.fit(X_train, y_train)` puis `.predict(X_test)` ; `accuracy_score(y_test, predictions)` ; `colonnes[arbre.tree_.feature[0]]`.

</details>

In [ ]:
# À toi
arbre = None
predictions = None
precision_arbre = None
premiere_question = None
print("Précision de l'arbre :", precision_arbre, "· première question sur :", premiere_question)

In [ ]:
verifier("Exercice 5 · arbre entraîné", isinstance(arbre, DecisionTreeClassifier) and arbre.get_depth() == 3)
verifier("Exercice 5 · précision", precision_arbre is not None and round(float(precision_arbre), 3) == 0.976)
verifier("Exercice 5 · première question", premiere_question == "flipper_length_mm")

<details><summary>Solution</summary>

```python
arbre = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
predictions = arbre.predict(X_test)
precision_arbre = accuracy_score(y_test, predictions)
premiere_question = colonnes[arbre.tree_.feature[0]]
print("Précision de l'arbre :", precision_arbre, "· première question sur :", premiere_question)
plt.figure(figsize=(16, 7))
plot_tree(arbre, feature_names=colonnes, class_names=arbre.classes_, filled=True, fontsize=9)
plt.show()
```

</details>

## Exercice 6 ⭐⭐ · La matrice de confusion

Où l'arbre se trompe-t-il ? Construis `matrice`, le tableau croisé « vraie espèce × espèce prédite » (`pd.crosstab`),
compte `nb_erreurs` (les cases hors de la diagonale) et trouve `espece_confondue` : l'espèce que l'arbre rate le plus.

Résultat attendu : 2 erreurs seulement, toutes sur la même espèce.

<details><summary>Indice</summary>

`pd.crosstab(y_test, predictions, rownames=["Vraie"], colnames=["Prédite"])` ; la diagonale : `np.trace(matrice.values)` ; `(y_test != predictions)` est vrai sur les erreurs, et `y_test[erreurs].value_counts().idxmax()`.

</details>

In [ ]:
# À toi
matrice = None
nb_erreurs = None
espece_confondue = None
print(matrice)
print("Erreurs :", nb_erreurs, "· espèce la plus confondue :", espece_confondue)

In [ ]:
verifier("Exercice 6 · matrice", matrice is not None and int(np.trace(np.asarray(matrice))) == 82)
verifier("Exercice 6 · erreurs", nb_erreurs == 2)
verifier("Exercice 6 · espèce confondue", espece_confondue == "Chinstrap")

<details><summary>Solution</summary>

```python
matrice = pd.crosstab(y_test, predictions, rownames=["Vraie"], colnames=["Prédite"])
nb_erreurs = int(len(y_test) - np.trace(matrice.values))
erreurs = y_test != predictions
espece_confondue = y_test[erreurs].value_counts().idxmax()
print(matrice)
print("Erreurs :", nb_erreurs, "· espèce la plus confondue :", espece_confondue)   # 2 Chinstrap pris pour des Adelie
```

</details>

## Exercice 7 ⭐⭐ · Les voisins et l'échelle

Entraîne un k-NN à 5 voisins et mesure `precision_knn`. Puis mets toutes les colonnes à la même échelle avec un
`StandardScaler` (ajusté sur le train, appliqué au train **et** au test), réentraîne et mesure `precision_knn_echelle`.
Conclus dans `meilleur` : `"avec"` ou `"sans"` mise à l'échelle ?

Résultat attendu : le poids en grammes écrasait les mesures en millimètres ; avec l'échelle, k-NN fait presque un sans-faute.

<details><summary>Indice</summary>

`echelle = StandardScaler().fit(X_train)` puis `echelle.transform(X_train)` et `echelle.transform(X_test)`.

</details>

In [ ]:
# À toi
precision_knn = None
precision_knn_echelle = None
meilleur = "..."   # "avec" ou "sans"
print("k-NN sans échelle :", precision_knn, "· avec échelle :", precision_knn_echelle, "→ meilleur :", meilleur)

In [ ]:
verifier("Exercice 7 · k-NN brut", precision_knn is not None and round(float(precision_knn), 2) == 0.86)
verifier("Exercice 7 · k-NN mis à l'échelle", precision_knn_echelle is not None and round(float(precision_knn_echelle), 2) == 0.99)
verifier("Exercice 7 · conclusion", meilleur == "avec")

<details><summary>Solution</summary>

```python
voisins = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
precision_knn = accuracy_score(y_test, voisins.predict(X_test))
echelle = StandardScaler().fit(X_train)
voisins_echelle = KNeighborsClassifier(n_neighbors=5).fit(echelle.transform(X_train), y_train)
precision_knn_echelle = accuracy_score(y_test, voisins_echelle.predict(echelle.transform(X_test)))
meilleur = "avec"
print("k-NN sans échelle :", precision_knn, "· avec échelle :", precision_knn_echelle, "→ meilleur :", meilleur)
```

</details>

## Exercice 8 ⭐⭐ · Apprendre par cœur

Pour des profondeurs de 1 à 15, entraîne un arbre (`random_state=42`) et range dans `prec_train` et `prec_test`
sa précision sur le train et sur le test. Trouve `profondeur_par_coeur` (la première profondeur où le train atteint
100 %) et `meilleure_profondeur_test` (la plus petite profondeur qui donne la meilleure précision de test).
Trace les deux courbes.

Résultat attendu : le train finit à 100 %, le test plafonne bien avant : c'est le sur-apprentissage.

<details><summary>Indice</summary>

`for p in range(1, 16):` ; `prec_train.index(1.0) + 1` ; `prec_test.index(max(prec_test)) + 1`.

</details>

In [ ]:
# À toi
profondeurs = list(range(1, 16))
prec_train, prec_test = [], []
profondeur_par_coeur = None
meilleure_profondeur_test = None
print("Par cœur à partir de la profondeur", profondeur_par_coeur, "· meilleure profondeur en test :", meilleure_profondeur_test)

In [ ]:
verifier("Exercice 8 · quinze arbres", len(prec_train) == 15 and len(prec_test) == 15 and prec_train[-1] == 1.0)
verifier("Exercice 8 · par cœur", profondeur_par_coeur == 7)
verifier("Exercice 8 · meilleure profondeur", meilleure_profondeur_test == 2)

<details><summary>Solution</summary>

```python
profondeurs = list(range(1, 16))
prec_train, prec_test = [], []
for p in profondeurs:
    a = DecisionTreeClassifier(max_depth=p, random_state=42).fit(X_train, y_train)
    prec_train.append(accuracy_score(y_train, a.predict(X_train)))
    prec_test.append(accuracy_score(y_test, a.predict(X_test)))
profondeur_par_coeur = prec_train.index(1.0) + 1
meilleure_profondeur_test = prec_test.index(max(prec_test)) + 1
print("Par cœur à partir de la profondeur", profondeur_par_coeur, "· meilleure profondeur en test :", meilleure_profondeur_test)
plt.figure(figsize=(8, 4))
plt.plot(profondeurs, prec_train, marker="o", label="train"); plt.plot(profondeurs, prec_test, marker="s", label="test")
plt.xlabel("Profondeur"); plt.ylabel("Précision"); plt.xticks(profondeurs); plt.legend()
plt.title("Le train monte à 100 %, le test plafonne dès la profondeur 2"); plt.show()
```

</details>

## Exercice 9 ⭐⭐ · Prédire un nombre

Changeons de famille : deviner le **poids** (`body_mass_g`) à partir des 3 mesures du bec et de la nageoire.
Coupe les données (`test_size=0.25`, `random_state=42`), entraîne une `LinearRegression` et mesure `erreur_regression`
(l'erreur absolue moyenne, en grammes). Compare avec `erreur_bete` : l'erreur d'un modèle qui prédit toujours le poids
moyen du train.

Résultat attendu : la régression divise l'erreur par plus de deux.

<details><summary>Indice</summary>

`mean_absolute_error(yp_test, predictions)` ; pour le modèle bête, prédire `[yp_train.mean()] * len(yp_test)`.

</details>

In [ ]:
# À toi
colonnes_poids = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm"]
erreur_regression = None
erreur_bete = None
print("Erreur de la régression :", erreur_regression, "g · modèle bête :", erreur_bete, "g")

In [ ]:
verifier("Exercice 9 · régression", erreur_regression is not None and abs(erreur_regression - 295.5) < 1)
verifier("Exercice 9 · modèle bête", erreur_bete is not None and abs(erreur_bete - 658) < 1)
verifier("Exercice 9 · deux fois mieux", erreur_regression is not None and erreur_bete is not None and erreur_regression < erreur_bete / 2)

<details><summary>Solution</summary>

```python
colonnes_poids = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm"]
Xp_train, Xp_test, yp_train, yp_test = train_test_split(manchots[colonnes_poids], manchots["body_mass_g"], test_size=0.25, random_state=42)
regression = LinearRegression().fit(Xp_train, yp_train)
erreur_regression = mean_absolute_error(yp_test, regression.predict(Xp_test))
erreur_bete = mean_absolute_error(yp_test, [yp_train.mean()] * len(yp_test))
print("Erreur de la régression :", erreur_regression, "g · modèle bête :", erreur_bete, "g")
```

</details>

## Exercice 10 ⭐⭐⭐ · Le piège des classes déséquilibrées

Sur les Pokémon, prédis `Legendary` à partir des 6 `stats`. Coupe avec `test_size=0.25`, `random_state=42` et
`stratify=y_leg` (pour garder la même part de légendaires des deux côtés). Calcule `precision_toujours_non` (un modèle
qui répond « pas légendaire » à tout le monde), entraîne `arbre_leg` (profondeur 3, `random_state=42`) et mesure
`precision_arbre_leg`. Puis compte `legendaires_total_test` (les vrais légendaires du test) et `legendaires_retrouves`
(ceux que l'arbre a bien prédits).

Résultat attendu : le modèle bête et l'arbre ont la **même** précision... mais l'un des deux retrouve quelques légendaires.

<details><summary>Indice</summary>

`(y_leg_test == False).mean()` ; les retrouvés : `((pred_leg == True) & (y_leg_test == True)).sum()`.

</details>

In [ ]:
# À toi
X_leg = pokemon[stats]
y_leg = pokemon["Legendary"]
precision_toujours_non = None
arbre_leg = None
precision_arbre_leg = None
legendaires_total_test = None
legendaires_retrouves = None
print("Toujours non :", precision_toujours_non, "· arbre :", precision_arbre_leg)
print("Légendaires retrouvés :", legendaires_retrouves, "/", legendaires_total_test)

In [ ]:
verifier("Exercice 10 · toujours non", precision_toujours_non is not None and round(float(precision_toujours_non), 2) == 0.92)
verifier("Exercice 10 · arbre", precision_arbre_leg is not None and round(float(precision_arbre_leg), 2) == 0.92)
verifier("Exercice 10 · légendaires du test", legendaires_total_test == 16)
verifier("Exercice 10 · légendaires retrouvés", legendaires_retrouves == 3)

<details><summary>Solution</summary>

```python
X_leg = pokemon[stats]
y_leg = pokemon["Legendary"]
X_leg_train, X_leg_test, y_leg_train, y_leg_test = train_test_split(X_leg, y_leg, test_size=0.25, random_state=42, stratify=y_leg)
precision_toujours_non = (y_leg_test == False).mean()
arbre_leg = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_leg_train, y_leg_train)
pred_leg = arbre_leg.predict(X_leg_test)
precision_arbre_leg = accuracy_score(y_leg_test, pred_leg)
legendaires_total_test = int(y_leg_test.sum())
legendaires_retrouves = int(((pred_leg == True) & (y_leg_test == True)).sum())
print("Toujours non :", precision_toujours_non, "· arbre :", precision_arbre_leg)
print("Légendaires retrouvés :", legendaires_retrouves, "/", legendaires_total_test)
print(pd.crosstab(y_leg_test, pred_leg, rownames=["Vrai"], colnames=["Prédit"]))
```

</details>

## Exercice 11 ⭐⭐⭐ · Mesurer ce qui compte : le rappel

Quand une classe est rare, on regarde le **rappel** : la part des vrais légendaires que le modèle retrouve
(`retrouvés / total`). Calcule `rappel_3` pour l'arbre de profondeur 3, puis entraîne un arbre de profondeur 5 et
calcule `precision_5` et `rappel_5`. Quelle profondeur choisis-tu pour un modèle qui doit **trouver** les légendaires ?
Réponds dans `meilleure_profondeur`.

Résultat attendu : la précision bouge à peine, le rappel fait ×3.

<details><summary>Indice</summary>

`rappel = retrouves / total` ; refais les mêmes calculs qu'à l'exercice 10 avec `max_depth=5`.

</details>

In [ ]:
# À toi
rappel_3 = None
precision_5 = None
rappel_5 = None
meilleure_profondeur = None
print("Profondeur 3 → rappel", rappel_3, "· profondeur 5 → précision", precision_5, ", rappel", rappel_5)
print("Je choisis la profondeur", meilleure_profondeur)

In [ ]:
verifier("Exercice 11 · rappel profondeur 3", rappel_3 is not None and round(float(rappel_3), 2) == 0.19)
verifier("Exercice 11 · précision profondeur 5", precision_5 is not None and round(float(precision_5), 3) == 0.925)
verifier("Exercice 11 · rappel profondeur 5", rappel_5 is not None and round(float(rappel_5), 2) == 0.56)
verifier("Exercice 11 · choix", meilleure_profondeur == 5)

<details><summary>Solution</summary>

```python
rappel_3 = legendaires_retrouves / legendaires_total_test
arbre_5 = DecisionTreeClassifier(max_depth=5, random_state=42).fit(X_leg_train, y_leg_train)
pred_5 = arbre_5.predict(X_leg_test)
precision_5 = accuracy_score(y_leg_test, pred_5)
rappel_5 = int(((pred_5 == True) & (y_leg_test == True)).sum()) / legendaires_total_test
meilleure_profondeur = 5
print("Profondeur 3 → rappel", rappel_3, "· profondeur 5 → précision", precision_5, ", rappel", rappel_5)
print("Je choisis la profondeur", meilleure_profondeur)   # 92 % → 92,5 % de précision, mais 3 → 9 légendaires retrouvés sur 16
```

</details>

## Exercice 12 ⭐⭐⭐ · Défi : la recette complète sur trois types

Les stats suffisent-elles à deviner le **type** d'un Pokémon ? Garde seulement les types `Water`, `Fire` et `Grass`
dans `trois`, prends les 6 `stats` comme `X3` et `Type 1` comme `y3`, coupe (`test_size=0.25`, `random_state=42`,
sans `stratify`), puis remplis `resultats` avec trois précisions : `"bête"` (le type le plus fréquent du train),
`"arbre"` (profondeur 3, `random_state=42`) et `"k-NN"` (5 voisins, avec `StandardScaler`). Conclus : `stats_suffisent`
(`True` / `False`) et une `phrase` qui explique ce que ton modèle a compris... ou pas.

Résultat attendu : aucun modèle ne bat le modèle bête. Un bon modèle ne peut pas deviner ce qui n'est pas dans les données !

<details><summary>Indice</summary>

`pokemon[pokemon["Type 1"].isin(["Water", "Fire", "Grass"])]` ; le reste est la recette des exercices 3 à 7.

</details>

In [ ]:
# À toi
trois = None
resultats = {"bête": None, "arbre": None, "k-NN": None}
stats_suffisent = None
phrase = "Mon modèle a compris que ..."
print(resultats)
print(phrase)

In [ ]:
verifier("Défi · trois types", isinstance(trois, pd.DataFrame) and len(trois) == 234)
verifier("Défi · modèle bête", resultats.get("bête") is not None and round(float(resultats["bête"]), 2) == 0.47)
verifier("Défi · arbre et k-NN mesurés", all(resultats.get(k) is not None and 0 <= float(resultats[k]) <= 1 for k in ["arbre", "k-NN"]))
verifier("Défi · conclusion", stats_suffisent is False and len(phrase) > 30 and "..." not in phrase)

<details><summary>Solution</summary>

```python
trois = pokemon[pokemon["Type 1"].isin(["Water", "Fire", "Grass"])]
X3 = trois[stats]
y3 = trois["Type 1"]
X3_train, X3_test, y3_train, y3_test = train_test_split(X3, y3, test_size=0.25, random_state=42)

resultats = {}
resultats["bête"] = (y3_test == y3_train.value_counts().index[0]).mean()
arbre3 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X3_train, y3_train)
resultats["arbre"] = accuracy_score(y3_test, arbre3.predict(X3_test))
echelle3 = StandardScaler().fit(X3_train)
knn3 = KNeighborsClassifier(n_neighbors=5).fit(echelle3.transform(X3_train), y3_train)
resultats["k-NN"] = accuracy_score(y3_test, knn3.predict(echelle3.transform(X3_test)))

stats_suffisent = False
phrase = "Mon modèle n'a rien compris d'utile : avec les seules stats, ni l'arbre ni k-NN ne font mieux que répondre toujours Water. Le type n'est pas écrit dans les stats."
print(resultats)
print(phrase)
```

</details>

## Bravo !

Tu as fait le tour du premier modèle : X et y, train / test, le modèle bête, l'arbre qu'on peut lire, k-NN et l'échelle,
le sur-apprentissage, une régression, et le piège des classes déséquilibrées (regarde le rappel, pas seulement la
précision). C'est exactement la recette que tu vas appliquer sur le Titanic aux séances 7 et 8.
